In [3]:
# Cell 1: Import Thư viện
%load_ext autoreload
%autoreload 2

import sys
import os
import pandas as pd
import numpy as np
import yfinance as yf
import joblib
import warnings

warnings.filterwarnings('ignore')
sys.path.append(os.path.abspath(".."))
from utils.feature_engineer import TechnicalFeatures

# Cell 2: Tải Mô hình AI đã huấn luyện
model_path = "../data/processed/xgboost_model.pkl"
if not os.path.exists(model_path):
    raise FileNotFoundError(f"❌ Không tìm thấy file mô hình tại {model_path}. Hãy chạy Notebook 07 trước!")

model = joblib.load(model_path)
expected_features = list(model.feature_names_in_)
print(f"✅ Đã nạp Mô hình XGBoost thành công! Mô hình yêu cầu {len(expected_features)} đặc trưng.")

# Cell 3: Tải dữ liệu thị trường mới nhất cho MBB
ticker = "MBB.VN" 
print(f"🔄 Đang tải dữ liệu thời gian thực cho cổ phiếu {ticker}...")

# Tải 250 phiên gần nhất để đủ dữ liệu tính các chỉ báo chu kỳ dài (SMA50, SMA200...)
df_live = yf.download(ticker, period="250d", interval="1d", progress=False)

if df_live.empty:
    raise ValueError(f"❌ Không thể tải dữ liệu cho mã {ticker}. Hãy kiểm tra lại kết nối mạng hoặc Ticker!")

# Xử lý triệt để MultiIndex của yfinance bản mới
if isinstance(df_live.columns, pd.MultiIndex):
    df_live.columns = df_live.columns.get_level_values(0)

# Cell 4: Tạo Đặc trưng (Feature Engineering) & Sửa lỗi thiếu cột
te_live = TechnicalFeatures(df_live)
df_live_features = te_live.generate_all_features()

# TỰ ĐỘNG BÙ ĐẮP DỮ LIỆU CỘT THIẾU (Ví dụ: SMA_50)
missing_cols = [col for col in expected_features if col not in df_live_features.columns]
if missing_cols:
    for col in missing_cols:
        if col == 'SMA_50' and 'Close' in df_live_features.columns:
            df_live_features['SMA_50'] = df_live_features['Close'].rolling(window=50, min_periods=1).mean()
        else:
            df_live_features[col] = 0.0

# Lọc bỏ các dòng NaN do tính toán chỉ báo lag/chu kỳ
df_clean_live = df_live_features.dropna(subset=expected_features)

if df_clean_live.empty:
    raise ValueError("❌ Dữ liệu sau khi tính đặc trưng bị rỗng! Cần thêm số lượng phiên giao dịch.")

# Lấy dòng dữ liệu MỚI NHẤT (Phiên giao dịch hôm nay)
today_data = df_clean_live.iloc[[-1]].copy()

latest_date = pd.to_datetime(today_data.index[0]).strftime("%Y-%m-%d")
latest_close_price = float(today_data['Close'].values[0])

# Trích xuất đúng các cột đặc trưng theo đúng thứ tự mô hình yêu cầu
X_live = today_data[expected_features]

# Cell 5: AI Dự đoán & Tính Vùng Giá Mua/Bán ĐỘNG bằng ATR (Quant Standard)
p_buy = model.predict_proba(X_live)[0][1]

BUY_ENTRY = 0.42   
SELL_EXIT = 0.25   

close_price = latest_close_price

# 1. Tính toán giá trị ATR 14 ngày từ dữ liệu sống
if 'ATR' in df_clean_live.columns:
    atr_val = df_clean_live['ATR'].iloc[-1]
else:
    # Tự tính ATR nếu chưa có sẵn cột
    high_low = df_clean_live['High'] - df_clean_live['Low']
    atr_val = high_low.rolling(14).mean().iloc[-1]

# 2. Bội số ATR cho Quản trị rủi ro (R:R = 1 : 2)
SL_ATR_MULT = 1.5   # Cắt lỗ = 1.5 lần ATR
TP_ATR_MULT = 3.0   # Chốt lời = 3.0 lần ATR

# 3. Tính toán các mốc giá động
buy_price_min = close_price - (0.2 * atr_val)
buy_price_max = close_price + (0.2 * atr_val)
stop_loss_price = close_price - (SL_ATR_MULT * atr_val)
target_price = close_price + (TP_ATR_MULT * atr_val)

# Tính % tương ứng để người dùng dễ theo dõi
sl_pct = ((close_price - stop_loss_price) / close_price) * 100
tp_pct = ((target_price - close_price) / close_price) * 100

# In Báo cáo Kế hoạch Giao dịch Tự động
print("\n" + "★"*65)
print(f"   🤖 BÁO CÁO TÍN HIỆU AI & KẾ HOẠCH GIAO DỊCH ĐỘNG (ATR) - MÃ: {ticker}")
print("★"*65)
print(f" 📅 Phiên cập nhật      : {latest_date}")
print(f" 💵 Giá đóng cửa (ATC)  : {close_price:,.0f} VNĐ")
print(f" 📉 Độ biến động ATR14  : {atr_val:,.0f} VNĐ / phiên")
print(f" 📊 Độ tự tin P(Buy)    : {p_buy * 100:.2f}% (Ngưỡng Mua: {BUY_ENTRY*100:.0f}%)")
print("-" * 65)

if p_buy >= BUY_ENTRY:
    print(" 🟢 KHUYẾN NGHỊ: MUA / TIẾP TỤC NẮM GIỮ (BULLISH)")
    print("\n 🎯 KẾ HOẠCH GIAO DỊCH TỰ ĐỘNG THEO BIẾN ĐỘNG (ATR):")
    print(f"    • Vùng giá Mua khớp lệnh : {buy_price_min:,.0f} - {buy_price_max:,.0f} VNĐ")
    print(f"    • Giá Chốt Lời (TP)       : {target_price:,.0f} VNĐ (+{tp_pct:.1f}%)")
    print(f"    • Giá Cắt Lỗ   (SL)       : {stop_loss_price:,.0f} VNĐ (-{sl_pct:.1f}%)")
    print(f"    • Tỷ lệ Lời / Lỗ (R:R)    : 1 : {(tp_pct/sl_pct):.1f}")
elif p_buy < SELL_EXIT:
    print(" 🔴 KHUYẾN NGHỊ: BÁN / ĐỨNG NGOÀI (BEARISH)")
    print("\n 🎯 KẾ HOẠCH GIAO DỊCH:")
    print(f"    • Giá Bán khuyến nghị     : {close_price:,.0f} VNĐ")
    print("    • Quản trị tài sản        : Đưa tỷ trọng cổ phiếu về 0%, giữ tiền mặt.")
else:
    print(" 🟡 KHUYẾN NGHỊ: THEO DÕI (NEUTRAL)")
    print("\n 🎯 KẾ HOẠCH GIAO DỊCH:")
    print(f"    • Mốc Cắt Lỗ bảo vệ hàng cũ : {stop_loss_price:,.0f} VNĐ (-{sl_pct:.1f}%)")

print("★"*65)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✅ Đã nạp Mô hình XGBoost thành công! Mô hình yêu cầu 10 đặc trưng.
🔄 Đang tải dữ liệu thời gian thực cho cổ phiếu MBB.VN...


2026-07-31 17:51:01,959 [INFO] Bắt đầu tính toán Technical Indicators & Macro Features...
2026-07-31 17:51:02,428 [INFO] Hoàn tất. Tổng số features (bao gồm vĩ mô): 21
2026-07-31 17:51:02,429 [INFO] Đã loại bỏ 49 dòng NaN ở đầu chuỗi dữ liệu.



★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
   🤖 BÁO CÁO TÍN HIỆU AI & KẾ HOẠCH GIAO DỊCH ĐỘNG (ATR) - MÃ: MBB.VN
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
 📅 Phiên cập nhật      : 2026-07-31
 💵 Giá đóng cửa (ATC)  : 22,500 VNĐ
 📉 Độ biến động ATR14  : 540 VNĐ / phiên
 📊 Độ tự tin P(Buy)    : 85.43% (Ngưỡng Mua: 42%)
-----------------------------------------------------------------
 🟢 KHUYẾN NGHỊ: MUA / TIẾP TỤC NẮM GIỮ (BULLISH)

 🎯 KẾ HOẠCH GIAO DỊCH TỰ ĐỘNG THEO BIẾN ĐỘNG (ATR):
    • Vùng giá Mua khớp lệnh : 22,392 - 22,608 VNĐ
    • Giá Chốt Lời (TP)       : 24,119 VNĐ (+7.2%)
    • Giá Cắt Lỗ   (SL)       : 21,690 VNĐ (-3.6%)
    • Tỷ lệ Lời / Lỗ (R:R)    : 1 : 2.0
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
